In [1]:
import numpy as np
import pandas as pd
import os
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF

import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [3]:
folder = "Data"

data = {}

for i in range(1, 9):

    file_path = os.path.join(
        folder,
        f"function_{i}.csv"
    )

    data[i] = pd.read_csv(file_path)

    print(
        f"Function {i}: "
        f"{len(data[i])} observations, "
        f"{len(data[i].columns) - 1} dimensions"
    )

Function 1: 11 observations, 2 dimensions
Function 2: 11 observations, 2 dimensions
Function 3: 16 observations, 3 dimensions
Function 4: 31 observations, 4 dimensions
Function 5: 21 observations, 4 dimensions
Function 6: 21 observations, 5 dimensions
Function 7: 31 observations, 6 dimensions
Function 8: 41 observations, 8 dimensions


In [4]:
# Round 2 query points

round2_points = {
    1: [0.999973, 0.999843],
    2: [0.655286, 0.929896],
    3: [0.397530, 0.090477, 0.447319],
    4: [0.480756, 0.474000, 0.229059, 0.429178],
    5: [0.514842, 0.856105, 0.961506, 0.998492],
    6: [0.246549, 0.344645, 0.479525, 0.733976, 0.025332],
    7: [0.080334, 0.317465, 0.485517, 0.055569, 0.406883, 0.931976],
    8: [0.078127, 0.668711, 0.054188, 0.209488,
        0.863552, 0.717799, 0.074439, 0.866300]
}

round2_outputs = {
    1: 1.868495640628433e-192,
    2: 0.36601049236039945,
    3: -0.06086980027965498,
    4: -3.743172382958963,
    5: 2765.8197868846855,
    6: -0.45851061638506707,
    7: 0.975232656186371,
    8: 9.62112231085
}

# Add the new observations
for i in range(1, 9):

    new_row = pd.DataFrame(
        [round2_points[i] + [round2_outputs[i]]],
        columns=data[i].columns
    )

    data[i] = pd.concat(
        [data[i], new_row],
        ignore_index=True
    )

# Confirm updated counts
for i in range(1, 9):
    print(f"Function {i}: {len(data[i])} observations")

Function 1: 12 observations
Function 2: 12 observations
Function 3: 17 observations
Function 4: 32 observations
Function 5: 22 observations
Function 6: 22 observations
Function 7: 32 observations
Function 8: 42 observations


In [5]:
# Assess the performance of the latest queries

for i in range(1, 9):

    # Exclude the latest observation to find the previous best
    previous_data = data[i].iloc[:-1]

    previous_best = previous_data["output"].max()
    new_output = data[i].iloc[-1]["output"]

    difference = new_output - previous_best
    updated_best = data[i]["output"].max()

    print("=" * 60)
    print(f"FUNCTION {i}")
    print("=" * 60)

    print(f"Previous best: {previous_best:.6f}")
    print(f"New output:    {new_output:.6f}")
    print(f"Difference:    {difference:.6f}")
    print(f"Updated best:  {updated_best:.6f}")

    if new_output > previous_best:
        print("Result: NEW BEST")
    else:
        print("Result: Did not improve")

    print()

FUNCTION 1
Previous best: 0.000000
New output:    0.000000
Difference:    -0.000000
Updated best:  0.000000
Result: Did not improve

FUNCTION 2
Previous best: 0.611205
New output:    0.366010
Difference:    -0.245195
Updated best:  0.611205
Result: Did not improve

FUNCTION 3
Previous best: -0.020167
New output:    -0.060870
Difference:    -0.040703
Updated best:  -0.020167
Result: Did not improve

FUNCTION 4
Previous best: -0.300645
New output:    -3.743172
Difference:    -3.442528
Updated best:  -0.300645
Result: Did not improve

FUNCTION 5
Previous best: 2329.015854
New output:    2765.819787
Difference:    436.803933
Updated best:  2765.819787
Result: NEW BEST

FUNCTION 6
Previous best: -0.714265
New output:    -0.458511
Difference:    0.255754
Updated best:  -0.458511
Result: NEW BEST

FUNCTION 7
Previous best: 1.950612
New output:    0.975233
Difference:    -0.975380
Updated best:  1.950612
Result: Did not improve

FUNCTION 8
Previous best: 9.821986
New output:    9.621122
Differ

In [6]:
# Refit the Gaussian Process for each function

models_week3 = {}

for i in range(1, 9):

    df = data[i]

    # Separate inputs and outputs
    X = df.drop(columns="output").values
    y = df["output"].values

    # RBF kernel
    kernel = RBF(
        length_scale=1.0,
        length_scale_bounds=(1e-5, 1e5)
    )

    # Gaussian Process
    gp = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        random_state=42,
        n_restarts_optimizer=10
    )

    # Fit model
    gp.fit(X, y)

    models_week3[i] = gp

    print("=" * 60)
    print(f"FUNCTION {i}")
    print("=" * 60)
    print("Fitted kernel:", gp.kernel_)
    print()

FUNCTION 1
Fitted kernel: RBF(length_scale=1e-05)

FUNCTION 2
Fitted kernel: RBF(length_scale=0.0427)

FUNCTION 3
Fitted kernel: RBF(length_scale=0.202)

FUNCTION 4
Fitted kernel: RBF(length_scale=0.495)

FUNCTION 5
Fitted kernel: RBF(length_scale=0.413)

FUNCTION 6
Fitted kernel: RBF(length_scale=1e-05)

FUNCTION 7
Fitted kernel: RBF(length_scale=0.244)

FUNCTION 8
Fitted kernel: RBF(length_scale=0.931)



In [7]:
# Check prediction and uncertainty variation for Functions 1 and 6

rng = np.random.default_rng(42)

diagnostic_candidates = {}

for i in [1, 6]:

    df = data[i]
    gp = models_week3[i]

    n_dimensions = len(df.columns) - 1

    # Generate 100,000 candidate points
    X_candidates = rng.uniform(
        0,
        1,
        size=(100000, n_dimensions)
    )

    diagnostic_candidates[i] = X_candidates

    # GP predictions
    mean, std = gp.predict(
        X_candidates,
        return_std=True
    )

    print("=" * 60)
    print(f"FUNCTION {i}")
    print("=" * 60)

    print(f"Predicted mean minimum: {mean.min():.10f}")
    print(f"Predicted mean maximum: {mean.max():.10f}")
    print(f"Predicted mean range:   {mean.max() - mean.min():.10f}")

    print()

    print(f"Uncertainty minimum:    {std.min():.10f}")
    print(f"Uncertainty maximum:    {std.max():.10f}")
    print(f"Uncertainty range:      {std.max() - std.min():.10f}")

    print()

FUNCTION 1
Predicted mean minimum: -0.0003005052
Predicted mean maximum: -0.0000001770
Predicted mean range:   0.0003003282

Uncertainty minimum:    0.0000342068
Uncertainty maximum:    0.0009966631
Uncertainty range:      0.0009624562

FUNCTION 6
Predicted mean minimum: -1.5012665979
Predicted mean maximum: -1.5012665979
Predicted mean range:   0.0000000000

Uncertainty minimum:    0.5421470063
Uncertainty maximum:    0.5421470063
Uncertainty range:      0.0000000000



In [8]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score

# Create high/low performance classes for each function

svm_data = {}

for i in range(1, 9):

    df = data[i]

    X = df.drop(columns="output").values
    y = df["output"].values

    # Median used as threshold
    threshold = np.median(y)

    # 1 = above median, 0 = at or below median
    y_class = (y > threshold).astype(int)

    svm_data[i] = {
        "X": X,
        "y_class": y_class,
        "threshold": threshold
    }

    print("=" * 60)
    print(f"FUNCTION {i}")
    print("=" * 60)
    print(f"Median threshold: {threshold:.6f}")
    print(f"Low-performing observations:  {(y_class == 0).sum()}")
    print(f"High-performing observations: {(y_class == 1).sum()}")
    print()

FUNCTION 1
Median threshold: 0.000000
Low-performing observations:  6
High-performing observations: 6

FUNCTION 2
Median threshold: 0.229792
Low-performing observations:  6
High-performing observations: 6

FUNCTION 3
Median threshold: -0.094190
Low-performing observations:  9
High-performing observations: 8

FUNCTION 4
Median threshold: -15.756741
Low-performing observations:  16
High-performing observations: 16

FUNCTION 5
Median threshold: 64.431793
Low-performing observations:  11
High-performing observations: 11

FUNCTION 6
Median threshold: -1.446370
Low-performing observations:  11
High-performing observations: 11

FUNCTION 7
Median threshold: 0.088196
Low-performing observations:  16
High-performing observations: 16

FUNCTION 8
Median threshold: 7.940817
Low-performing observations:  21
High-performing observations: 21



In [9]:
# Compare Linear and RBF SVM performance

for i in range(1, 9):

    X = svm_data[i]["X"]
    y_class = svm_data[i]["y_class"]

    linear_svm = make_pipeline(
        StandardScaler(),
        SVC(kernel="linear")
    )

    rbf_svm = make_pipeline(
        StandardScaler(),
        SVC(kernel="rbf")
    )

    linear_scores = cross_val_score(
        linear_svm,
        X,
        y_class,
        cv=3,
        scoring="accuracy"
    )

    rbf_scores = cross_val_score(
        rbf_svm,
        X,
        y_class,
        cv=3,
        scoring="accuracy"
    )

    print("=" * 60)
    print(f"FUNCTION {i}")
    print("=" * 60)

    print(f"Linear SVM mean accuracy: {linear_scores.mean():.3f}")
    print(f"RBF SVM mean accuracy:    {rbf_scores.mean():.3f}")

    print()

FUNCTION 1
Linear SVM mean accuracy: 0.167
RBF SVM mean accuracy:    0.500

FUNCTION 2
Linear SVM mean accuracy: 0.667
RBF SVM mean accuracy:    0.833

FUNCTION 3
Linear SVM mean accuracy: 0.344
RBF SVM mean accuracy:    0.522

FUNCTION 4
Linear SVM mean accuracy: 0.627
RBF SVM mean accuracy:    0.782

FUNCTION 5
Linear SVM mean accuracy: 0.649
RBF SVM mean accuracy:    0.458

FUNCTION 6
Linear SVM mean accuracy: 0.821
RBF SVM mean accuracy:    0.768

FUNCTION 7
Linear SVM mean accuracy: 0.785
RBF SVM mean accuracy:    0.815

FUNCTION 8
Linear SVM mean accuracy: 0.857
RBF SVM mean accuracy:    0.833



In [10]:
# Summarise the evidence used to reassess each strategy

strategy_summary = pd.DataFrame({
    "Function": range(1, 9),

    "Latest_Query_Improved": [
        False, False, False, False,
        True, True, False, False
    ],

    "GP_Usable": [
        False, True, True, True,
        True, False, True, True
    ],

    "Linear_SVM_Accuracy": [
        0.167, 0.667, 0.344, 0.627,
        0.649, 0.821, 0.785, 0.857
    ],

    "RBF_SVM_Accuracy": [
        0.500, 0.833, 0.522, 0.782,
        0.458, 0.768, 0.815, 0.833
    ],

    "Proposed_Strategy": [
        "Exploration",
        "Balanced UCB",
        "Balanced UCB",
        "UCB / moderate exploitation",
        "Greater exploitation",
        "SVM-informed / exploration",
        "Balanced UCB",
        "Balanced UCB"
    ]
})

print(strategy_summary.to_string(index=False))

 Function  Latest_Query_Improved  GP_Usable  Linear_SVM_Accuracy  RBF_SVM_Accuracy           Proposed_Strategy
        1                  False      False                0.167             0.500                 Exploration
        2                  False       True                0.667             0.833                Balanced UCB
        3                  False       True                0.344             0.522                Balanced UCB
        4                  False       True                0.627             0.782 UCB / moderate exploitation
        5                   True       True                0.649             0.458        Greater exploitation
        6                   True      False                0.821             0.768  SVM-informed / exploration
        7                  False       True                0.785             0.815                Balanced UCB
        8                  False       True                0.857             0.833                Balanced UCB


In [11]:
# Select UCB candidates for functions with usable GP models

kappa = 2.5
n_candidates = 100000

ucb_functions = [2, 3, 4, 5, 7, 8]

ucb_selected = {}

rng = np.random.default_rng(42)

for i in ucb_functions:

    df = data[i]
    gp = models_week3[i]

    n_dimensions = len(df.columns) - 1

    # Generate candidate points across the full search space
    X_candidates = rng.uniform(
        0,
        1,
        size=(n_candidates, n_dimensions)
    )

    # Predict mean and uncertainty
    mean, std = gp.predict(
        X_candidates,
        return_std=True
    )

    # Calculate Upper Confidence Bound
    ucb = mean + kappa * std

    # Select candidate with highest UCB
    best_index = np.argmax(ucb)
    selected_point = X_candidates[best_index]

    ucb_selected[i] = selected_point

    print("=" * 60)
    print(f"FUNCTION {i}")
    print("=" * 60)

    print("Selected point:")
    print(selected_point)

    print(f"Predicted output: {mean[best_index]:.6f}")
    print(f"Uncertainty:      {std[best_index]:.6f}")
    print(f"UCB:              {ucb[best_index]:.6f}")

    print()

FUNCTION 2
Selected point:
[0.70962749 0.88195252]
Predicted output: 0.449335
Uncertainty:      0.185666
UCB:              0.913500

FUNCTION 3
Selected point:
[0.73203545 0.99729984 0.01565321]
Predicted output: -0.076907
Uncertainty:      0.077556
UCB:              0.116983

FUNCTION 4
Selected point:
[0.3919759  0.42475707 0.49482291 0.46442447]
Predicted output: -0.064270
Uncertainty:      0.420976
UCB:              0.988170

FUNCTION 5
Selected point:
[0.67162912 0.98111794 0.99045818 0.98518882]
Predicted output: 2631.682259
Uncertainty:      240.331386
UCB:              3232.510725

FUNCTION 7
Selected point:
[0.02154929 0.31333367 0.36021627 0.14173151 0.29532786 0.68654814]
Predicted output: 1.662117
Uncertainty:      0.258204
UCB:              2.307626

FUNCTION 8
Selected point:
[0.02552911 0.45123759 0.09820892 0.03139148 0.4124209  0.13857332
 0.11493429 0.99436062]
Predicted output: 9.635913
Uncertainty:      0.434286
UCB:              10.721628



In [12]:
# Compare UCB-selected points with the current best observations

for i in ucb_functions:

    df = data[i]

    input_columns = [col for col in df.columns if col != "output"]

    # Find current best observation
    best_index = df["output"].idxmax()

    current_best_point = df.loc[
        best_index,
        input_columns
    ].values.astype(float)

    current_best_output = df.loc[
        best_index,
        "output"
    ]

    proposed_point = ucb_selected[i]

    # Euclidean distance
    distance = np.linalg.norm(
        proposed_point - current_best_point
    )

    print("=" * 60)
    print(f"FUNCTION {i}")
    print("=" * 60)

    print("Current best point:")
    print(np.round(current_best_point, 6))

    print(f"Current best output: {current_best_output:.6f}")

    print("\nProposed new point:")
    print(np.round(proposed_point, 6))

    print(f"\nDistance from current best: {distance:.4f}")

    print()

FUNCTION 2
Current best point:
[0.702637 0.926564]
Current best output: 0.611205

Proposed new point:
[0.709627 0.881953]

Distance from current best: 0.0452

FUNCTION 3
Current best point:
[0.409499 0.399    0.519487]
Current best output: -0.020167

Proposed new point:
[0.732035 0.9973   0.015653]

Distance from current best: 0.8461

FUNCTION 4
Current best point:
[0.399157 0.433203 0.357699 0.461233]
Current best output: -0.300645

Proposed new point:
[0.391976 0.424757 0.494823 0.464424]

Distance from current best: 0.1376

FUNCTION 5
Current best point:
[0.514842 0.856105 0.961506 0.998492]
Current best output: 2765.819787

Proposed new point:
[0.671629 0.981118 0.990458 0.985189]

Distance from current best: 0.2030

FUNCTION 7
Current best point:
[0.120627 0.403921 0.379973 0.146533 0.335853 0.763305]
Current best output: 1.950612

Proposed new point:
[0.021549 0.313334 0.360216 0.141732 0.295328 0.686548]

Distance from current best: 0.1612

FUNCTION 8
Current best point:
[0.2433

In [13]:
from sklearn.metrics import pairwise_distances

# Function 1 - maximum-distance exploration

i = 1

X_existing = data[i].drop(columns="output").values

rng = np.random.default_rng(42)

X_candidates = rng.uniform(
    0,
    1,
    size=(100000, X_existing.shape[1])
)

# Distance from every candidate to every existing observation
distances = pairwise_distances(
    X_candidates,
    X_existing
)

# Find each candidate's nearest existing observation
nearest_distances = distances.min(axis=1)

# Select the candidate whose nearest observation is furthest away
best_index = np.argmax(nearest_distances)

f1_selected = X_candidates[best_index]

print("=" * 60)
print("FUNCTION 1 - MAXIMUM-DISTANCE EXPLORATION")
print("=" * 60)

print("Selected point:")
print(f1_selected)

print(
    "\nDistance to nearest existing observation:",
    nearest_distances[best_index]
)

FUNCTION 1 - MAXIMUM-DISTANCE EXPLORATION
Selected point:
[4.50627487e-05 3.06208459e-03]

Distance to nearest existing observation: 0.32167082653265755


In [14]:
# Function 6 - SVM-informed candidate selection

i = 6

X = svm_data[i]["X"]
y_class = svm_data[i]["y_class"]

# Fit the linear SVM using all available observations
f6_svm = make_pipeline(
    StandardScaler(),
    SVC(kernel="linear")
)

f6_svm.fit(X, y_class)

# Generate candidate points
rng = np.random.default_rng(42)

X_candidates = rng.uniform(
    0,
    1,
    size=(100000, X.shape[1])
)

# Decision score:
# larger positive values = stronger classification as high-performing
decision_scores = f6_svm.decision_function(X_candidates)

best_index = np.argmax(decision_scores)

f6_svm_candidate = X_candidates[best_index]

print("=" * 60)
print("FUNCTION 6 - SVM-INFORMED CANDIDATE")
print("=" * 60)

print("Selected point:")
print(f6_svm_candidate)

print(
    "\nSVM decision score:",
    decision_scores[best_index]
)

# Compare with current best
best_obs_index = data[6]["output"].idxmax()

current_best = (
    data[6]
    .drop(columns="output")
    .loc[best_obs_index]
    .values
    .astype(float)
)

distance = np.linalg.norm(
    f6_svm_candidate - current_best
)

print("\nCurrent best point:")
print(current_best)

print(
    "\nDistance from current best:",
    distance
)

FUNCTION 6 - SVM-INFORMED CANDIDATE
Selected point:
[0.01375812 0.14792993 0.44559686 0.99484501 0.01346948]

SVM decision score: 5.272052074088469

Current best point:
[0.246549 0.344645 0.479525 0.733976 0.025332]

Distance from current best: 0.4027814448701828


In [15]:
# Compare the F6 SVM candidate with existing high-performing observations

i = 6

X = svm_data[i]["X"]
y_class = svm_data[i]["y_class"]

# Existing observations classified as high-performing
X_high = X[y_class == 1]

# Distance from SVM candidate to each high-performing observation
high_distances = np.linalg.norm(
    X_high - f6_svm_candidate,
    axis=1
)

print("=" * 60)
print("FUNCTION 6 - SVM CANDIDATE CHECK")
print("=" * 60)

print(
    f"Distance to nearest high-performing observation: "
    f"{high_distances.min():.4f}"
)

print(
    f"Average distance to high-performing observations: "
    f"{high_distances.mean():.4f}"
)

print(
    f"Number of existing high-performing observations: "
    f"{len(X_high)}"
)

FUNCTION 6 - SVM CANDIDATE CHECK
Distance to nearest high-performing observation: 0.4028
Average distance to high-performing observations: 0.8865
Number of existing high-performing observations: 11


In [16]:
# Combine all Week 3 proposed query points

final_points = {
    1: f1_selected,
    2: ucb_selected[2],
    3: ucb_selected[3],
    4: ucb_selected[4],
    5: ucb_selected[5],
    6: f6_svm_candidate,
    7: ucb_selected[7],
    8: ucb_selected[8]
}

selection_methods = {
    1: "Maximum-distance exploration",
    2: "RBF GP + UCB",
    3: "RBF GP + UCB",
    4: "RBF GP + UCB",
    5: "RBF GP + UCB",
    6: "Linear SVM",
    7: "RBF GP + UCB",
    8: "RBF GP + UCB"
}

for i in range(1, 9):

    point = np.asarray(final_points[i])
    X_existing = data[i].drop(columns="output").values

    # Check bounds
    in_bounds = np.all((point >= 0) & (point <= 1))

    # Distance to existing observations
    distances = np.linalg.norm(
        X_existing - point,
        axis=1
    )

    nearest_distance = distances.min()

    # Duplicate check
    duplicate = np.any(
        np.all(
            np.isclose(X_existing, point),
            axis=1
        )
    )

    print("=" * 60)
    print(f"FUNCTION {i}")
    print("=" * 60)

    print("Selected point:")
    print(np.round(point, 6))

    print(f"Selection method: {selection_methods[i]}")
    print(f"In bounds: {in_bounds}")
    print(f"Duplicate existing point: {duplicate}")
    print(f"Distance to nearest observation: {nearest_distance:.6f}")

    if i == 1:
        max_distance_correct = np.allclose(
            point,
            f1_selected
        )
        print(
            f"Maximum-distance selection correct: "
            f"{max_distance_correct}"
        )

    print()

FUNCTION 1
Selected point:
[4.500e-05 3.062e-03]
Selection method: Maximum-distance exploration
In bounds: True
Duplicate existing point: False
Distance to nearest observation: 0.321671
Maximum-distance selection correct: True

FUNCTION 2
Selected point:
[0.709627 0.881953]
Selection method: RBF GP + UCB
In bounds: True
Duplicate existing point: False
Distance to nearest observation: 0.045156

FUNCTION 3
Selected point:
[0.732035 0.9973   0.015653]
Selection method: RBF GP + UCB
In bounds: True
Duplicate existing point: False
Distance to nearest observation: 0.306635

FUNCTION 4
Selected point:
[0.391976 0.424757 0.494823 0.464424]
Selection method: RBF GP + UCB
In bounds: True
Duplicate existing point: False
Distance to nearest observation: 0.137608

FUNCTION 5
Selected point:
[0.671629 0.981118 0.990458 0.985189]
Selection method: RBF GP + UCB
In bounds: True
Duplicate existing point: False
Distance to nearest observation: 0.203041

FUNCTION 6
Selected point:
[0.013758 0.14793  0.445

In [17]:
for i in range(1, 9):
    submission = "-".join(
        f"{value:.6f}"
        for value in final_points[i]
    )

    print(f"Function {i}: {submission}")

Function 1: 0.000045-0.003062
Function 2: 0.709627-0.881953
Function 3: 0.732035-0.997300-0.015653
Function 4: 0.391976-0.424757-0.494823-0.464424
Function 5: 0.671629-0.981118-0.990458-0.985189
Function 6: 0.013758-0.147930-0.445597-0.994845-0.013469
Function 7: 0.021549-0.313334-0.360216-0.141732-0.295328-0.686548
Function 8: 0.025529-0.451238-0.098209-0.031391-0.412421-0.138573-0.114934-0.994361
